In [ ]:
import torch 
import torch.nn.functional as F 
import torch.nn as nn 


import os, random
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt

from utils_cbs import normalize, denormalize, score_normalize, score_denormalize, calculate_psnr, calculate_ssim, compute_batch_metrics, log_image
from configs.vp import AI4Scup2_ddpm_continuous as configs
from cbs_model import ConvergentBornSeries_Batch, ConvergentBornSeries_Batch_Adjoint
from image_datasets import USCT_Dataset_CBS

config = configs.get_config()
config.device_ids = [0]
config.device = torch.device('cuda:' + str(config.device_ids[0])) if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
# Setting Measurement_Mode Parameters
file = {
    "max_output" : 1595.1279, 
    "min_output" : 1408.692, 
    "resize_size": (256,256), 
    "measurement_mode": "sparse", # "sparse", "partial"， "sparse_2", "partial_2"
    "stage": "eval", # "train", "eval" 
    "frequency": "500k", # "500k"
    "base_mask": np.load("auxiliary_data/mask.npy"), # 256 * 256 测量域相邻 Detectors 的屏蔽 
    "x_pos":loadmat('auxiliary_data/x_pos.mat')['x_pos256'],
    "y_pos":loadmat('auxiliary_data/y_pos.mat')['y_pos256'],
    }

if file["measurement_mode"] == "sparse" or file["measurement_mode"] == "sparse_2":
    file["base_dir_dobs_500k_train"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_sparse/dobs_500k/train"
    file["base_dir_dobs_500k_eval"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_sparse/dobs_500k/eval"
    file["base_dir_speed_train"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_sparse/speed/train"
    file["base_dir_speed_eval"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_sparse/speed/eval"
    
    file["mask"] = file["base_mask"][0::4,0::4]
    if  file["measurement_mode"] == "sparse":
        file["downsampled_input_mask"] = file["mask"]
        file["receiver_indices"] = torch.cat((torch.tensor(file["x_pos"][0::4].astype(np.int64)), torch.tensor(file["y_pos"][0::4].astype(np.int64))), dim = 1)
        file["transmitter_indices"] = torch.cat((torch.tensor(file["x_pos"][0::4].astype(np.int64)), torch.tensor(file["y_pos"][0::4].astype(np.int64))), dim = 1)
    
    elif file["measurement_mode"] == "sparse_2":
        file["downsampled_input_mask"] = file["mask"][::2, ::2]
        file["receiver_indices"] = torch.cat((torch.tensor(file["x_pos"][0::8].astype(np.int64)), torch.tensor(file["y_pos"][0::8].astype(np.int64))), dim = 1)
        file["transmitter_indices"] = torch.cat((torch.tensor(file["x_pos"][0::8].astype(np.int64)), torch.tensor(file["y_pos"][0::8].astype(np.int64))), dim = 1)
    
elif file["measurement_mode"] == "partial" or file["measurement_mode"] == "partial_2":
    file["base_dir_dobs_500k_train"] = "/home/caoxiang/Desktop/Dataset/Datasets_download/AI4Scup2_simulated_CBS_partial/dobs_500k/train"
    file["base_dir_dobs_500k_eval"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_partial/dobs_500k/eval"
    file["base_dir_speed_train"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_partial/speed/train"
    file["base_dir_speed_eval"] = "/home/caoxiang/Desktop/Datasets/Datasets_download/AI4Scup2_simulated_CBS_partial/speed/eval"
    
    base_valid_rows, base_valid_cols = [i for i in range(64)], [j for j in range(128,128+64)] 
    file["mask"] = file["base_mask"][base_valid_rows,:][:,base_valid_cols]
    if  file["measurement_mode"] == "partial":
        file["downsampled_input_mask"] = file["mask"]
        valid_rows, valid_cols = [i for i in range(64)], [j for j in range(128,128+64)] 
        file["receiver_indices"] = torch.cat((torch.tensor(file["x_pos"][valid_cols].astype(np.int64)), torch.tensor(file["y_pos"][valid_cols].astype(np.int64))), dim = 1)
        file["transmitter_indices"] = torch.cat((torch.tensor(file["x_pos"][valid_rows].astype(np.int64)), torch.tensor(file["y_pos"][valid_rows].astype(np.int64))), dim = 1)
    
    elif file["measurement_mode"] == "partial_2":
        file["downsampled_input_mask"] = file["mask"][[i for i in range(32)], :][:, [j for j in range(32)]]
        valid_rows, valid_cols = [i for i in range(32)], [j for j in range(128,128+32)] 
        file["receiver_indices"] = torch.cat((torch.tensor(file["x_pos"][valid_cols].astype(np.int64)), torch.tensor(file["y_pos"][valid_cols].astype(np.int64))), dim = 1)
        file["transmitter_indices"] = torch.cat((torch.tensor(file["x_pos"][valid_rows].astype(np.int64)), torch.tensor(file["y_pos"][valid_rows].astype(np.int64))), dim = 1)

eval_dataset = USCT_Dataset_CBS(data_dict=file)

# 加载预重建网络（有监督）& 否则不要加载 （无监督）

In [ ]:
# from InversionNet_modules.Baselines_modified import InversionNet
# if file["measurement_mode"] == "partial": 
#     InversionNet_model_path = "/home/caoxiang/Desktop/full_waveform_inversion/InversionNet_baseline/logs/exp_500k_partial/checkpoints/checkpoint_1000.pth"
# elif file["measurement_mode"] == "partial_2":
#     InversionNet_model_path = "/home/caoxiang/Desktop/full_waveform_inversion/InversionNet_baseline/logs/exp_500k_partial_2/checkpoints/checkpoint_1000.pth"
# elif file["measurement_mode"] == "sparse":
#     InversionNet_model_path = "/home/caoxiang/Desktop/full_waveform_inversion/InversionNet_baseline/logs/exp_500k_sparse/checkpoints/checkpoint_1000.pth"
# elif file["measurement_mode"] == "sparse_2":
#     InversionNet_model_path = "/home/caoxiang/Desktop/full_waveform_inversion/InversionNet_baseline/logs/exp_500k_sparse_2/checkpoints/checkpoint_1000.pth"

# # Load Checkpoint File
# InversionNet_model_state = torch.load(InversionNet_model_path, map_location=config.device)
# # Initialize Model & Load Checkpoint
# InversionNet_model = InversionNet().cuda(device=config.device_ids[0])
# InversionNet_model.load_state_dict(InversionNet_model_state['model'], strict=False)
# _ = InversionNet_model.eval()

In [ ]:
step = 0 # 指定对应sample的编号～
input_batch, output_batch = eval_dataset[step]

cbs_batch_one = 1500 * torch.ones_like(denormalize(output_batch.unsqueeze(0))).to(config.device)
output_cbs_batch = denormalize(output_batch.unsqueeze(0)).to(config.device)

# IN_pred_batch = InversionNet_model(input_batch.to(config.device))
# IN_pred_cbs_batch = denormalize(IN_pred_batch).to(config.device)

if file["measurement_mode"] == "sparse":
    downsampled_dobs_500k_batch = torch.complex(input_batch[:,0,:,:], input_batch[:,1,:,:]).to(config.device)
elif file["measurement_mode"] == "sparse_2":
    downsampled_dobs_500k_batch = torch.complex(input_batch[:,0,:,:], input_batch[:,1,:,:])[...,:32,:32].to(config.device)
elif file["measurement_mode"] == "partial":
    downsampled_dobs_500k_batch = torch.complex(input_batch[:,0,:,:], input_batch[:,1,:,:]).to(config.device)
elif file["measurement_mode"] == "partial_2":
    downsampled_dobs_500k_batch = torch.complex(input_batch[:,0,:,:], input_batch[:,1,:,:])[...,:32,:32] .to(config.device)
elif file["measurement_mode"] == "full":
    downsampled_dobs_500k_batch = torch.complex(input_batch[:,0,:,:], input_batch[:,1,:,:]).to(config.device)

In [ ]:
plt.figure()
plt.imshow(output_cbs_batch[0,0,90:390,90:390].cpu(), cmap="inferno", vmin=1408.692, vmax=1595.1279)
_ = plt.axis('off')
plt.show()
plt.close()


plt.figure()
plt.imshow(input_batch.numpy()[1,0])
_ = plt.axis('off')
plt.show()
plt.close()

In [ ]:
def add_awgn(signal, snr_dB):
    signal_stack = torch.stack((signal.real, signal.imag))
    signal_power = torch.mean(signal_stack ** 2)
    
    snr_linear = 10**(snr_dB / 10.0)
    noise_power = signal_power / snr_linear
    
    noise = torch.sqrt(noise_power) * torch.randn_like(signal_stack)
    noisy_signal_stack = signal_stack + noise
    noisy_signal = torch.complex(noisy_signal_stack[0], noisy_signal_stack[1])
    return noisy_signal

def downsample_noise_operator(u_batch, receiver_indices, downsampled_mask, snr_dB_list = []):
    downsampled_mask = torch.tensor(downsampled_mask).to(u_batch.device)
    clean_signal = u_batch[0, :, receiver_indices[:,0], receiver_indices[:,1]] * downsampled_mask
    noised_signal_list = [clean_signal]
    for snr_dB in snr_dB_list:
        noised_signal = add_awgn(clean_signal, snr_dB)
        noised_signal_list.append(noised_signal * downsampled_mask)
    return  torch.stack(noised_signal_list, dim=0)

In [ ]:
# Update transmitter and receiver indices for full measurement mode
file["transmitter_indices"] = torch.cat((torch.tensor(file["x_pos"].astype(np.int64)), torch.tensor(file["y_pos"].astype(np.int64))), dim = 1)
file["receiver_indices"] = torch.cat((torch.tensor(file["x_pos"].astype(np.int64)), torch.tensor(file["y_pos"].astype(np.int64))), dim = 1)
file["downsampled_input_mask"] = file["base_mask"]

# 测试 Convergent Born Series 算子

In [ ]:
model_500k_output_cbs_batch =  ConvergentBornSeries_Batch(
            f = 500e+3,
            sos= output_cbs_batch, 
            boundary_width=[300,300],
            boundary_strength=225,
            boundary_type='PML3',
            src_loc_set= file["transmitter_indices"], 
            device=config.device)

u_500k_output_cbs_batch = model_500k_output_cbs_batch(max_iters=500)
downsampled_dobs_500k_output_cbs_batch = downsample_noise_operator(u_500k_output_cbs_batch, file["receiver_indices"].to(config.device), file["downsampled_input_mask"], [10,5])

In [ ]:
plt.imshow(torch.real(downsampled_dobs_500k_output_cbs_batch)[0].cpu())
print(downsampled_dobs_500k_output_cbs_batch.shape)

# 加载 Score_Model

In [ ]:
import sde_lib
from models import ddpm as ddpm_model
from models import utils as mutils
from configs.vp import AI4Scup2_ddpm_continuous as configs

In [ ]:
score_model_path = "/home/caoxiang/Desktop/diffusion_model/score_sde_pytorch_training/workdir/AI4Scup2/256/checkpoints-meta/checkpoint.pth"

score_model = ddpm_model.DDPM(config)
score_model = torch.nn.DataParallel(score_model, config.device_ids)
score_model = score_model.cuda(device=config.device_ids[0])

# Resume training when intermediate checkpoints are detected; 
loaded_state = torch.load(score_model_path, map_location=config.device)
score_model.load_state_dict(loaded_state['model'], strict=False)

# Define the diffusion process SDE
sde = sde_lib.VPSDE(beta_min=config.model.beta_min, beta_max=config.model.beta_max, N=config.model.num_scales)
score_fn = mutils.get_score_fn(sde, score_model, train=False, continuous=config.training.continuous)
sampling_eps = 1e-4

# 测试 Adjoint 算子

In [ ]:
class ConvergentBornSeries_Batch_Adjoint(ConvergentBornSeries_Batch):
    def __init__(self, batch_model, rec_loc, dobs_500k_batch, dobs_500k_mask):
        self.__dict__ = batch_model.__dict__.copy() # 复制定义好的 batch_model 的参数 
        # masked_dobs_500k_batch 输入是 [N, 32/64, 32/64] 复数型  的 masked_noised_observation_input
        
        self.dobs_500k_batch = dobs_500k_batch.to(batch_model.device)
        self.dobs_500k_mask = torch.tensor(dobs_500k_mask).to(batch_model.device) # [M, L]
        self.rec_loc = rec_loc # [L, 2] 是在 [480, 480] 图上的坐标点～

    def forward(self, u_batch, max_iters=5):
        # sos -> k -> k2_pad 参数不变，当前状态输入 recevier 处的 measurement difference 是 y - A(x)
        # rec_loc_batch 应该是 [32/64, 2] 的位置坐标和前面是一样的, u_batch 是 [N, M, 480, 480] 的声压分布图像 (M = 32/64)
        self.batch_size, self.sample_size = u_batch.shape[0], u_batch.shape[1]
        self.rec_diff_batch, rec_diff_value = self.compute_rec_diff(u_batch) # N, M, 480, 480 （每个 [480, 480] 中对应的稀疏
        self.adjoint_lamb = torch.zeros_like(self.k2_pad[0:1,:,:,:]).to(self.device) # [N, 1, 512, 512]
        self.adjoint_lamb_batch = self.adjoint_lamb.repeat(self.batch_size, self.sample_size, 1, 1) # [N, M, 512, 512]

        with torch.no_grad():
            for _ in range(max_iters):
                tmp1 = self.V * self.adjoint_lamb_batch
                tmp1[:, :, self.roi[0][0]:self.roi[0][1], self.roi[1][0]:self.roi[1][1]] += torch.conj(self.rec_diff_batch) # 在这一步修改残差！ 
                tmp2 = (self.gamma) * (self.adjoint_lamb_batch - torch.fft.ifftn(self.g0 * torch.fft.fftn(tmp1, dim=(-2, -1)), dim=(-2, -1)))
                self.adjoint_lamb_batch = self.adjoint_lamb_batch - tmp2

        self.real_inner_prod_sum = torch.real(torch.sum(self.adjoint_lamb_batch[:, :, self.roi[0][0]:self.roi[0][1], self.roi[1][0]:self.roi[1][1]] * u_batch, dim = 1))[:, None, :, :]# [N, M, 480, 480] -> [N, 1, 480, 480]
        self.sos_grad = -2 * (self.omega0**2/self.sos**3) * self.real_inner_prod_sum # [N, 1, 480, 480]

        return self.sos_grad, rec_diff_value

    def compute_rec_diff(self, u_batch):
        rec_diff_batch = torch.zeros_like(u_batch).to(self.device) # [N, M, 480, 480]
        
        self.rec_diff = - self.dobs_500k_batch + u_batch[:, :, self.rec_loc[:,0], self.rec_loc[:,1]] * self.dobs_500k_mask # y - A(x) 大小是 [N, M, L] (M = 32/64, L = 32/64)
        rec_diff_batch[:, :, self.rec_loc[:,0], self.rec_loc[:,1]] = self.rec_diff * (1e+7)
        
        return rec_diff_batch, torch.mean(torch.abs(self.rec_diff))

In [ ]:
model_500k_one_cbs_batch =  ConvergentBornSeries_Batch(
            f = 500e+3,
            sos= cbs_batch_one, 
            boundary_width=[300,300],
            boundary_strength=225,
            boundary_type='PML3',
            src_loc_set=file["transmitter_indices"], 
            device=config.device)

# model_500k_pred_cbs_batch =  ConvergentBornSeries_Batch(
#             f = 500e+3,
#             sos= IN_pred_cbs_batch, 
#             boundary_width=[300,300],
#             boundary_strength=225,
#             boundary_type='PML3',
#             src_loc_set=file["transmitter_indices"], 
#             device=config.device)

u_500k_one_cbs_batch = model_500k_one_cbs_batch(max_iters=500) # max_iters = 300 为界, max_iters=500 为佳
# u_500k_pred_cbs_batch = model_500k_pred_cbs_batch(max_iters=500) # max_iters = 300 为界, max_iters=500 为佳


model_500k_one_cbs_batch_grad = ConvergentBornSeries_Batch_Adjoint(
            batch_model = model_500k_one_cbs_batch, 
            rec_loc = file["receiver_indices"], 
            dobs_500k_batch = downsampled_dobs_500k_batch, 
            dobs_500k_mask = file["downsampled_input_mask"])

# model_500k_pred_cbs_batch_grad = ConvergentBornSeries_Batch_Adjoint(
#             batch_model = model_500k_pred_cbs_batch, 
#             rec_loc = file["receiver_indices"], 
#             dobs_500k_batch = downsampled_dobs_500k_batch, 
#             dobs_500k_mask = file["downsampled_input_mask"])

one_cbs_batch_grad, one_loss_value = model_500k_one_cbs_batch_grad(u_500k_one_cbs_batch.repeat(3,1,1,1), max_iters=500)
# pred_cbs_batch_grad, pred_loss_value = model_500k_pred_cbs_batch_grad(u_500k_pred_cbs_batch, max_iters=500)

In [ ]:
plt.imshow(one_cbs_batch_grad[2,0,90:390,90:390].cpu())

# DPS/DDS

In [ ]:
def dilate_mask(mask, kernel_size=3, device='cpu'):
    kernel = torch.ones(1, 1, kernel_size, kernel_size, device=device)
    mask = mask.float()
    dilated_mask = F.conv2d(mask, kernel, padding=kernel_size//2)
    dilated_mask = dilated_mask > 0

    return dilated_mask

In [ ]:
image_mask = (output_cbs_batch > 1500.1)|(output_cbs_batch < 1499.9)
dilated_mask = dilate_mask(image_mask, kernel_size=3, device=config.device)  # 使用 3x3 的

plt.figure()
plt.imshow(torch.clone(dilated_mask[0,0,90:390,90:390]).detach().cpu())
_ = plt.axis('off')
plt.show()
plt.close()

In [ ]:
import sde_lib
from models import ddpm as ddpm_model
from models import utils as mutils
from configs.vp import AI4Scup2_ddpm_continuous as configs

In [ ]:
score_model_path = "/home/caoxiang/Desktop/diffusion_model/score_sde_pytorch_training/workdir/AI4Scup2/256/checkpoints-meta/checkpoint.pth"

score_model = ddpm_model.DDPM(config)
score_model = torch.nn.DataParallel(score_model, config.device_ids)
score_model = score_model.cuda(device=config.device_ids[0])

# Resume training when intermediate checkpoints are detected; 
loaded_state = torch.load(score_model_path, map_location=config.device)
score_model.load_state_dict(loaded_state['model'], strict=False)

# Define the diffusion process SDE
sde = sde_lib.VPSDE(beta_min=config.model.beta_min, beta_max=config.model.beta_max, N=config.model.num_scales)
score_fn = mutils.get_score_fn(sde, score_model, train=False, continuous=config.training.continuous)
sampling_eps = 1e-4

In [ ]:
def neural_cbs_gradient_descent_DPS(pred_batch, image_mask, downsampled_input_batch, downsampled_input_mask, ds = 0.25, cbs_num_epoch= 1):
    pred_cbs_batch = denormalize(pred_batch)
    
    for epoch in range(cbs_num_epoch):
        model_500k_pred_cbs_batch =  ConvergentBornSeries_Batch(
            f = 500e+3,
            sos= pred_cbs_batch, 
            boundary_width=[300,300],
            boundary_strength=225,
            boundary_type='PML3',
            src_loc_set= file["transmitter_indices"] , 
            device=config.device)

        model_500k_pred_cbs_batch_grad = ConvergentBornSeries_Batch_Adjoint(
            batch_model = model_500k_pred_cbs_batch, 
            rec_loc = file["receiver_indices"], 
            dobs_500k_batch = downsampled_input_batch, 
            dobs_500k_mask = downsampled_input_mask) 
            
        u_500k_pred_cbs_batch = model_500k_pred_cbs_batch(max_iters=500)
        pred_cbs_batch_grad, loss_value = model_500k_pred_cbs_batch_grad(u_500k_pred_cbs_batch, max_iters=500)
        
        print("Epoch: " + str(epoch) + " , Loss: " + str(loss_value))
        pred_cbs_batch[image_mask] -= (ds/torch.sqrt(loss_value)) * pred_cbs_batch_grad[image_mask]
                
        plt.figure()
        plt.imshow(torch.clone(pred_cbs_batch[0,0,90:390,90:390]).detach().cpu(), cmap="inferno", vmin=1408.692, vmax=1595.1279)
        _ = plt.axis('off')
        plt.show()
        plt.close()


    pred_batch_after_cbs = normalize(pred_cbs_batch)
    return pred_batch_after_cbs

In [ ]:
def ancestral_sampling(pred_batch, dilated_mask, downsampled_input_batch, downsampled_input_mask, ts_value=[0.1, 0.08, 0.06, 0.04, 0.02, 0.01, 0.001], eta = 1.0, MRSDE_sampling = False):
    """
    Postprocessing: pred_batch 预重建图, ts_value[0] = 0.1 提供 known prior distribution, ts_value[1:-1] 提供采样（Grad + Noise + Tweedie)， ts_value[-1] = 0.001 提供额外去噪, eta = 1.0/0.0 控制 DDPM/DDIM 采样方式;
    MRSDE_sampling = True/False  控制 MRSDE 采样方式; 
    DPS: pred_batch 全1未知初始, ts_value 是 np.linspace(1,0,1000), eta = 1.0 控制 DDPM 采样方式; (唯一区别在于grad的更新是DDS的方式)
    DDS: pred_batch 全1未知初始, ts_value[0] = 1.0 提供完全 unknown prior distribution, ts_value[1:-1] 遵循 DDIM 的设定， ts_value[-1] = 0.001 提供额外去噪， eta = 0.0 控制 DDIM 采样方式；
    """
    # 本质上 Sampling 的过程 总结成 Tweedie Formula + Remixed noise (全新的噪声或者预测的噪声的加权);
    # 这两种极端的方式分别是 Ancestral_sampling in DDPM 和 Deterministic DDIM;
    # DDPM (Ancestral_Sampling) 和 Score SDE (Reverse SDE) 采样是等价的, 这种采样会改变采样轨迹，类似于 CT;
    # DDIM 和 Score SDE (Probability ODE) 的 x_t, x_0_hat 出发的采样是沿固定轨迹的，类似于 CD;
     
    # 初始化先验分布 ts_value[0]（CBS_grad + 随机噪声 + Tweedie 去噪)
    t_start = (ts_value[0]) * torch.ones(pred_batch.shape[0], device = pred_batch.device)
    t_start_i =  (torch.floor(t_start * sde.N) + 1)/sde.N
    # print("CBS~")
    x_0_hat_after_cbs = neural_cbs_gradient_descent_DPS(pred_batch, dilated_mask, downsampled_input_batch, downsampled_input_mask, ds = 0.1, cbs_num_epoch = 1)
    
    with torch.no_grad():    
        z = torch.randn_like(x_0_hat_after_cbs)
        mean, std, coef = sde.marginal_prob(x_0_hat_after_cbs, t_start_i)
        perturbed_data = mean + std[:, None, None, None] * z # 加噪这步来自于随机噪声就是 DDPM/加噪这步来自于上一步去噪噪声就是 DDIM 
        perturbed_score = score_fn(perturbed_data, t_start_i)

        # Tweedie Formula 
        x_0_hat_tweedie = (perturbed_data + perturbed_score * std[:, None, None, None] ** 2)/coef[:, None, None, None]
        
        # MRSDE Sampling
        if MRSDE_sampling == True: 
            x_0_hat = (coef) * x_0_hat_tweedie + (1 - coef) * pred_batch
        else: 
            x_0_hat = x_0_hat_tweedie
        
        print("Tweedie~")
        x_0_hat_plot = denormalize(x_0_hat).detach().cpu().numpy()
        
        plt.figure()
        plt.imshow(x_0_hat_plot[0,0,90:390,90:390], cmap="inferno", vmin=1408.692, vmax=1595.1279)
        _ = plt.axis('off')
        plt.show()
        plt.close()
    
    # 中间过程 dps 采样 （CBS_grad + 混合噪声 + Tweedie 去噪)
    for t_value in ts_value[1:]:
        t = (t_value) * torch.ones(x_0_hat.shape[0], device = x_0_hat.device)
        t_i =  (torch.floor(t * sde.N) + 1)/sde.N
        # print("CBS~")
        x_0_hat_after_cbs = neural_cbs_gradient_descent_DPS(x_0_hat, dilated_mask, downsampled_input_batch, downsampled_input_mask, ds = 0.1, cbs_num_epoch = 1)

        with torch.no_grad():
            z = torch.randn_like(x_0_hat_after_cbs)
            _, std_before, coef_before = torch.clone(mean).detach(), torch.clone(std).detach(), torch.clone(coef).detach()
            # std_before = \sqrt{1 - \bar{\alpha}_{t}}, coef_before = \sqrt{\bar{\alpha}_{t}}
            # 不能使用原来 sde.sde 中的beta系数，因为DDIM完全是通过 marginal_distribution 系数来构建的～
            mean, std, coef = sde.marginal_prob(x_0_hat_after_cbs, t_i)
            # 这里的 t_i 是在 DDS 的文章中为第t-1步 
            # std = \sqrt{1 - \bar{\alpha}_{t-1}}, coef = \sqrt{\bar{\alpha}_{t-1}} 
            beta_tilde_t = (std/std_before) * torch.sqrt(1 - (coef_before/coef)**2)
            deterministic_noise_weight = torch.sqrt(std**2 - (eta*beta_tilde_t)**2)
            # 加噪这步来自于混合噪声就是 DDPM / 加噪这步来自于上一步预测噪声就是 DDIM
            perturbed_noise_before = - torch.clone(perturbed_score).detach() * std_before[:, None, None, None]

            perturbed_data = mean + deterministic_noise_weight[:, None, None, None] * perturbed_noise_before + (eta*beta_tilde_t)[:, None, None, None] * z # DDPM 加噪到第t-1步～
            perturbed_score = score_fn(perturbed_data, t_i)
            # perturbed_score = - \frac{noise_{t-1}}{\sqrt{1 - \bar{\alpha}_{t-1}}}

            # Tweedie Formula 
            x_0_hat_tweedie = (perturbed_data + perturbed_score * std[:, None, None, None] ** 2)/coef[:, None, None, None]
            
            # MRSDE Sampling
            if MRSDE_sampling == True: 
                x_0_hat = (coef) * x_0_hat_tweedie + (1 - coef) * x_0_hat
            else: 
                x_0_hat = x_0_hat_tweedie

            print("Tweedie~")
            x_0_hat_plot = denormalize(x_0_hat).detach().cpu().numpy()
            print(x_0_hat_plot.shape)
            plt.figure()
            plt.imshow(x_0_hat_plot[0,0,90:390,90:390], cmap="inferno", vmin=1408.692, vmax=1595.1279)
            _ = plt.axis('off')
            plt.show()
            plt.close()
    
    # 最后一步仅去噪 (随机噪声 + Tweedie 去噪)
    with torch.no_grad():
        t_end = (ts_value[-1]) * torch.ones(x_0_hat.shape[0], device=x_0_hat.device)
        t_end_i =  (torch.floor(t_end * sde.N) + 1)/sde.N
        
        z = torch.randn_like(x_0_hat)
        mean, std, coef = sde.marginal_prob(x_0_hat, t_end_i)
        perturbed_data = mean + std[:, None, None, None] * z # 仅加噪～
        perturbed_score = score_fn(perturbed_data, t_end_i) # 
        
        # Tweedie Formula 
        x_0_hat_tweedie = (perturbed_data + perturbed_score * std[:, None, None, None] ** 2)/coef[:, None, None, None]

        # MRSDE Sampling
        if MRSDE_sampling == True: 
            x_0_hat = (coef) * x_0_hat_tweedie + (1 - coef) * x_0_hat
        else: 
            x_0_hat = x_0_hat_tweedie

    return x_0_hat

In [ ]:
IN_dps_batch = ancestral_sampling(torch.zeros_like(cbs_batch_one[1:2,:,:,:]), dilated_mask.repeat(1,1,1,1), downsampled_dobs_500k_batch[1:2,:,:], file["downsampled_input_mask"], ts_value = [value for value in np.linspace(0.99, 0.01, 50)], eta = 0.0)